# P2 · Fleet Maintenance Planning — Submission Candidate

This notebook adds a cost-optimised policy while retaining the existing model pipeline. It compares the new policy against the previous policies using nested, cross-fitted validation. The new additions are a permitted scikit-learn quantile-regression repair model and a direct expected-cost inspection/budget decision. The notebook implements the supplied competition objective:

- Inspection cost: 10 units.
- Parts shortage penalty: 0.20 per unit.
- Unused parts penalty: 0.02 per unit.
- Skipping inspection incurs the full realised repair cost when a vehicle fails.

The pipeline uses only permitted model families and permitted third-party libraries:
HistGradientBoosting, ElasticNet, logistic regression, Nystroem/kernel methods,
and optional CatBoost; NumPy, pandas, SciPy, and scikit-learn.

It includes:
- train/test sensor-shift diagnostics and the aligned-channel mapping;
- missing-value-aware sensor features;
- an audit of `followup_days`;
- out-of-fold risk and repair-severity predictions;
- cross-fitted policy evaluation using the official maintenance-cost formula;
- submission schema, row-order, flag, and budget checks.

**Execution status:** code cells have passed Python syntax parsing and the cost
function has hand-checks. The complete notebook has not been executed against
the competition CSVs in this environment. Run every cell from top to bottom
in the competition environment and inspect the validation-cost output before
uploading `submission.csv`. This is a candidate, not a claim of a verified
leaderboard improvement.


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import ks_2samp
from scipy.special import expit, logit
from scipy.linalg import cho_factor, cho_solve
from scipy.spatial.distance import cdist
from scipy.optimize import minimize

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import Nystroem

SEED = 42
INSPECTION, SHORTAGE, SURPLUS = 10.0, 0.20, 0.02
SENSORS = [f"s{i:03d}" for i in range(170)]
SCHEMA = ["id", "flag", "budget"]

# Set this to the directory containing the three competition CSV files.
# Use "." when train.csv, test.csv and sample_submission.csv are beside the notebook.
# In Kaggle, replace "." with the relevant /kaggle/input/<dataset-folder> path.
DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv", dtype={"id": str})
test = pd.read_csv(f"{DATA_DIR}/test.csv", dtype={"id": str})
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv", dtype={"id": str})

assert len(train) == 56_000 and len(test) == len(sample) == 20_000
assert list(sample) == SCHEMA
assert test.id.is_unique and sample.id.is_unique and set(test.id) == set(sample.id)
assert train.failure.isin([0, 1]).all()

failure = train.failure.to_numpy(dtype=int)
repair = train.repair_cost.fillna(0).to_numpy()
assert np.isfinite(repair).all()
assert np.array_equal(repair > 0, failure == 1)

print(f"{len(train):,} training vehicles, {failure.sum():,} recorded failures")
print(f"scikit-learn {sklearn.__version__}, NumPy {np.__version__}")

# CatBoost is explicitly permitted, but remains optional if unavailable.
USE_CATBOOST = True
try:
    if USE_CATBOOST:
        from catboost import CatBoostClassifier, CatBoostRegressor
except ImportError:
    USE_CATBOOST = False
    print("CatBoost is not installed; CatBoost challenger models are disabled.")


## Sensor readings and depot shift

The test channels `s055`–`s063` are aligned with training channels `s054`–`s062`.
The aligned test channels are appended as a separate block when building cost features.
We retain stable sensors for the main representation and explicitly map the aligned
channels to avoid accidental duplicate-name indexing.

In [ ]:
shift = {}
for name in SENSORS:
    old, new = train[name].dropna(), test[name].dropna()
    if min(old.nunique(), new.nunique()) < 2:
        shift[name] = 1.0
    else:
        distribution_shift = ks_2samp(old, new, method="asymp").statistic
        missingness_shift = abs(len(old) / len(train) - len(new) / len(test))
        shift[name] = max(distribution_shift, missingness_shift)

stable = [name for name in SENSORS if shift[name] < 0.10]
if not stable:
    raise ValueError("No stable sensors were retained; inspect the shift threshold.")

frequently_missing = train[stable].isna().mean().to_numpy() >= 0.20

def features(frame):
    readings = frame[stable].to_numpy(dtype="float32")
    missing = np.isnan(readings)
    counts = missing.sum(axis=1, dtype="float32")
    group_counts = missing[:, frequently_missing].sum(axis=1, dtype="float32")
    return np.column_stack((readings, counts, group_counts))

X, X_test = features(train), features(test)
print(f"Stable raw sensors: {len(stable)}")

scale = train[stable].quantile(0.90).to_numpy()
scale = np.where(np.isfinite(scale) & (scale > 0), scale, 1.0)
anchor_names = ["s071", "s123", "s149", "s053", "s012"]
anchor_cols = [stable.index(c) for c in anchor_names if c in stable]

source_channels = [f"s{i:03d}" for i in range(54, 63)]
target_channels = [f"s{i:03d}" for i in range(55, 64)]
resolution = np.array([3, 10, 10, 100, 100, 100, 10, 10, 10], dtype="float32")
channel_scale = train[source_channels].quantile(0.9).to_numpy(dtype="float32")
channel_scale = np.where(
    np.isfinite(channel_scale) & (channel_scale > 1), channel_scale, 1.0
).astype("float32")

def relative_features(frame, channels):
    raw = frame[stable].to_numpy(dtype="float32")
    scaled = raw / scale.astype("float32")

    if anchor_cols:
        exposure = np.nanmedian(scaled[:, anchor_cols], axis=1)
    else:
        exposure = np.nanmedian(scaled, axis=1)

    exposure = np.nan_to_num(exposure, nan=0.0, posinf=0.0, neginf=0.0)
    log_readings = np.log1p(np.maximum(scaled, 0))
    ratios = np.log1p(np.maximum(scaled, 0) / (0.01 + np.maximum(exposure[:, None], 0)))
    parts = [log_readings, ratios, features(frame)[:, -2:]]

    for start, end in [(23, 33), (37, 47), (80, 90), (109, 119), (129, 139), (154, 164)]:
        cols = [stable.index(f"s{i:03d}") for i in range(start, end) if f"s{i:03d}" in stable]
        if len(cols) < 2:
            continue
        block = scaled[:, cols]
        total = np.nansum(block, axis=1)
        parts.append(block / (0.01 + total[:, None]))

    block = frame[channels].to_numpy(dtype="float32")
    block = np.rint(block / resolution) * resolution / channel_scale
    parts.extend([
        np.log1p(np.maximum(block, 0)),
        np.log1p(np.maximum(block, 0) / (0.01 + np.maximum(exposure[:, None], 0)))
    ])
    return np.column_stack(parts).astype("float32")

Z, Z_test = relative_features(train, source_channels), relative_features(test, target_channels)

def active_channel_ratio(frame, name):
    active = frame[name].gt(100) & frame["s149"].gt(0)
    return frame.loc[active, name] / (1.0 + frame.loc[active, "s149"])

aligned_shift = [
    ks_2samp(
        active_channel_ratio(train, old),
        active_channel_ratio(test, new),
        method="asymp"
    ).statistic
    for old, new in zip(source_channels, target_channels)
]
print(f"Restored {len(aligned_shift)} channels. Largest active-ratio KS distance: {max(aligned_shift):.3f}")

## Follow-up and censoring audit
The dataset includes `followup_days` (capped at 365). Before interpreting `failure=0` as a true 365-day negative, check whether vehicles with shorter follow-up are right-censored. If they are, the current binary classifiers need a survival/IPCW treatment or should exclude censored negatives; a simple binary label is not enough.


In [ ]:
days = train["followup_days"].to_numpy(dtype=float)
assert np.isfinite(days).all(), "followup_days contains non-finite values"
assert ((days > 0) & (days <= 365)).all(), "Unexpected followup_days range"
short_nonfail = (failure == 0) & (days < 365)
full_nonfail = (failure == 0) & (days >= 365)
event_rows = failure == 1
print("Follow-up audit")
print(f"  Failures: {event_rows.sum():,}; median follow-up: {np.median(days[event_rows]):.1f} days")
print(f"  Non-failures with <365 days: {short_nonfail.sum():,} ({short_nonfail.mean():.2%} of all rows)")
print(f"  Non-failures with 365 days: {full_nonfail.sum():,}")
if short_nonfail.any():
    print("  CAUTION: if short non-failure follow-up means right-censoring, these rows are not confirmed 365-day negatives.")
else:
    print("  No short-follow-up non-failures detected; confirm followup_days semantics nonetheless.")


## Failure risk and repair severity models

The original HistGradientBoosting, relative-feature boosting, kernel-risk, ElasticNet
severity and kernel severity models are retained. CatBoost is added as an optional
challenger. Its prediction is evaluated through the same OOF pipeline; it should only
be retained if it improves the downstream maintenance objective.

In [ ]:
def fit_predict(fit_idx, query, seed):
    x, y, c = X[fit_idx], failure[fit_idx], repair[fit_idx]
    positive = y == 1

    cls_options = dict(
        learning_rate=0.075, max_iter=210, max_bins=127,
        early_stopping=False, random_state=seed
    )
    probabilities = []
    for leaves, min_leaf in ((7, 25), (15, 40)):
        clf = HistGradientBoostingClassifier(
            **cls_options, max_leaf_nodes=leaves, min_samples_leaf=min_leaf,
            l2_regularization=3.0, max_features=0.85
        )
        clf.fit(x, y)
        probabilities.append(clf.predict_proba(query)[:, 1])
    p = np.mean(probabilities, axis=0)

    mean_cost = c[positive].mean()
    weights = np.where(positive, c / mean_cost, 1.0)
    weighted = HistGradientBoostingClassifier(
        **cls_options, max_leaf_nodes=7, min_samples_leaf=35,
        l2_regularization=5.0, max_features=0.85
    )
    weighted.fit(x, y, sample_weight=weights)
    p_weighted = np.clip(weighted.predict_proba(query)[:, 1], 0, 0.985)
    expected_cost = mean_cost * p_weighted / (1 - p_weighted) * (1 - p)

    return np.column_stack((p, expected_cost))


def relative_predictions(indices, query, seed):
    x, y, c = Z[indices], failure[indices], repair[indices]
    positive = y == 1

    boost = HistGradientBoostingClassifier(
        max_leaf_nodes=12, min_samples_leaf=30, max_iter=175,
        max_bins=127, learning_rate=0.065, max_features=0.7,
        l2_regularization=6, early_stopping=False, random_state=seed
    )
    boost.fit(x, y, sample_weight=np.where(positive, 6.0, 1.0))
    score = boost.predict_proba(query)[:, 1]
    p_boost = score / (6.0 - 5.0 * score)

    def severity_features(rows):
        return np.column_stack((
            np.nan_to_num(rows, nan=0),
            np.isnan(rows),
            rows > 0
        ))

    scaler = StandardScaler()
    design = scaler.fit_transform(severity_features(x[positive]))
    model = ElasticNet(alpha=0.09, l1_ratio=0.55, max_iter=4000)
    model.fit(design, np.log(c[positive]))
    log_cost = model.predict(scaler.transform(severity_features(query)))
    log_cost = np.clip(log_cost, np.log(c[positive].min()), np.log(c.max()))
    return np.column_stack((p_boost, log_cost))


def catboost_risk(indices, query, seed):
    if not USE_CATBOOST:
        return None
    x = np.nan_to_num(Z[indices], nan=-0.75, posinf=0.0, neginf=0.0)
    q = np.nan_to_num(query, nan=-0.75, posinf=0.0, neginf=0.0)
    y = failure[indices]
    model = CatBoostClassifier(
        iterations=900, depth=6, learning_rate=0.035,
        loss_function="Logloss", eval_metric="Logloss",
        l2_leaf_reg=7, random_seed=seed, verbose=False,
        allow_writing_files=False, thread_count=-1
    )
    model.fit(x, y)
    return np.clip(model.predict_proba(q)[:, 1], 1e-6, 1 - 1e-6)


def catboost_repair_cost(raw_fit, costs, raw_query, seed):
    if not USE_CATBOOST:
        return None
    x = np.nan_to_num(raw_fit, nan=-0.75, posinf=0.0, neginf=0.0)
    q = np.nan_to_num(raw_query, nan=-0.75, posinf=0.0, neginf=0.0)
    y = np.log(costs)
    model = CatBoostRegressor(
        iterations=1200, depth=6, learning_rate=0.035,
        loss_function="RMSE", l2_leaf_reg=7,
        random_seed=seed, verbose=False,
        allow_writing_files=False, thread_count=-1
    )
    model.fit(x, y)
    return np.clip(model.predict(q), y.min(), y.max())


def kernel_risk(indices, query, seed):
    x = Z[indices]
    y = failure[indices]

    if "s071" in stable:
        active = x[:, stable.index("s071")]
    else:
        active = np.nanmedian(x[:, :len(stable)], axis=1)

    active = np.nan_to_num(active, nan=np.nanmedian(active))
    high = active >= np.nanquantile(active, 0.90)
    support = high | (y == 1)

    pool = np.nan_to_num(x[support], nan=-0.75)
    median = np.median(pool, axis=0)
    lo, hi = np.quantile(pool, [0.15, 0.85], axis=0)
    spread = np.maximum(hi - lo, 0.15)

    def standardized(raw):
        return np.clip(
            (np.nan_to_num(raw, nan=-0.75) - median) / spread,
            -4, 4
        )

    balanced = standardized(pool)
    positive = y[support] == 1
    a, b = balanced[positive], balanced[~positive]
    separation = (a.mean(axis=0) - b.mean(axis=0))**2 / (
        a.var(axis=0) + b.var(axis=0) + 0.02
    )
    select = np.argsort(separation)[-min(120, balanced.shape[1]):]

    def inputs(raw):
        return standardized(raw)[:, select]

    rng = np.random.default_rng(seed)
    difficult = np.flatnonzero(high & (y == 0))
    landmarks = np.r_[
        np.flatnonzero(y == 1),
        rng.choice(difficult, min(480, len(difficult)), replace=False)
    ]

    mapper = Nystroem(
        kernel="rbf", gamma=0.011,
        n_components=350, random_state=seed
    )
    mapper.fit(inputs(x[landmarks]))

    def transform(raw):
        return np.vstack([
            mapper.transform(inputs(raw[i:i+2500]))
            for i in range(0, len(raw), 2500)
        ])

    low = np.flatnonzero(~high & (y == 0))
    sampled = rng.choice(low, min(len(low), 6500), replace=False)
    rows = np.r_[np.flatnonzero(support), sampled]
    weights = np.where(
        y[rows] == 1, 8.0,
        np.where(high[rows], 1.0, len(low) / max(len(sampled), 1))
    )

    model = LogisticRegression(C=12, max_iter=350)
    model.fit(transform(x[rows]), y[rows], sample_weight=weights)
    score = model.predict_proba(transform(query))[:, 1]
    return score / (8 - 7 * score)

## Cost feature construction

The appended aligned channels must always be selected for sensor numbers 054–062.
For all other channels, the stable-sensor representation is used. This avoids the
previous `.index()` ambiguity when a sensor name appears twice.

In [ ]:
def cost_sensor_index(sensor_number):
    name = f"s{sensor_number:03d}"
    if 54 <= sensor_number <= 62:
        return len(stable) + source_channels.index(name)
    if name in stable:
        return stable.index(name)
    return None


sensor_blocks = [
    (23, 33), (37, 47), (54, 63),
    (80, 90), (109, 119), (129, 139), (154, 164)
]

cost_columns = stable + source_channels

bin_indices = [
    [
        idx for i in range(lo, hi)
        if (idx := cost_sensor_index(i)) is not None
    ]
    for lo, hi in sensor_blocks
]


def channel_shapes(raw):
    parts = []
    for lo, hi in sensor_blocks:
        cols = [
            idx for i in range(lo, hi)
            if (idx := cost_sensor_index(i)) is not None
        ]
        if len(cols) < 2:
            continue

        block = np.nan_to_num(raw[:, cols], nan=0.0)
        total = block.sum(axis=1, keepdims=True)
        share = block / np.maximum(total, 1.0)

        left = np.cumsum(share, axis=1)[:, :-1]
        right = np.cumsum(share[:, ::-1], axis=1)[:, -2::-1]
        ordinal = np.linspace(0, 1, len(cols))
        mean = share @ ordinal
        variance = np.maximum(0, share @ (ordinal ** 2) - mean ** 2)

        summary = np.column_stack((
            mean, np.sqrt(variance),
            (share ** 2).sum(axis=1), share.max(axis=1)
        ))
        parts.extend([
            summary, left,
            np.log1p(1000 * left),
            np.log1p(1000 * right)
        ])

    return np.column_stack(parts)


def cost_readings(frame, channels):
    raw = frame[stable].to_numpy(dtype=np.float64)
    matched = frame[channels].to_numpy(dtype=np.float64)
    return np.column_stack((raw, matched))


def cost_basis(raw, positive_scale):
    logged = np.log1p(np.maximum(raw, 0))
    relative = raw / positive_scale
    parts = [
        logged,
        np.log1p(10 * np.maximum(relative, 0)),
        np.arcsinh(relative),
        np.sqrt(np.clip(relative, 0, 25))
    ]

    for lo, hi in sensor_blocks:
        cols = [
            idx for i in range(lo, hi)
            if (idx := cost_sensor_index(i)) is not None
        ]
        if len(cols) < 2:
            continue
        block = np.nan_to_num(raw[:, cols], nan=0.0)
        total = block.sum(axis=1, keepdims=True)
        share = block / np.maximum(total, 1.0)
        parts.extend([share, np.sqrt(share), np.log1p(100 * share)])

    return np.column_stack(parts)


def fit_cost_models(raw_fit, z_fit, costs, raw_query, z_query):
    positive_scale = np.nanquantile(raw_fit, 0.75, axis=0)
    positive_scale = np.where(
        np.isfinite(positive_scale) & (positive_scale > 1),
        positive_scale, 1
    )

    def design(raw, z):
        parts = [
            z, np.isnan(z), z > 0,
            cost_basis(raw, positive_scale),
            channel_shapes(raw)
        ]
        return np.nan_to_num(
            np.column_stack(parts), nan=0, posinf=0, neginf=0
        ).astype(np.float64)

    scaler = StandardScaler()
    x = scaler.fit_transform(design(raw_fit, z_fit))
    log_cost = np.log(costs)

    first = ElasticNet(
        alpha=0.075, l1_ratio=0.8, max_iter=5000
    ).fit(x, log_cost)

    strength = np.abs(first.coef_)
    penalty = np.maximum(
        np.sqrt(strength / max(strength.max(), 1e-12)), 0.09
    )

    second = ElasticNet(
        alpha=0.035, l1_ratio=0.75, max_iter=5000
    ).fit(x * penalty, log_cost)

    coefficients = np.column_stack((
        first.coef_, second.coef_ * penalty
    ))
    intercepts = np.array([first.intercept_, second.intercept_])

    predictions = []
    for start in range(0, len(raw_query), 2500):
        batch = slice(start, start + 2500)
        values = scaler.transform(
            design(raw_query[batch], z_query[batch])
        )
        predictions.append(values @ coefficients + intercepts)

    return np.clip(
        np.vstack(predictions),
        np.log(costs.min()), np.log(costs.max())
    )


def utilization_features(raw):
    total = raw[:, bin_indices[0]].sum(axis=1)
    reference_names = [
        cost_columns[j] for j in bin_indices[0]
    ]
    typical = train[reference_names].sum(
        axis=1, min_count=len(reference_names)
    ).median()

    sensor_scale = np.maximum(
        train[cost_columns].quantile(0.9).to_numpy(), 1
    )
    relative = raw / sensor_scale
    pieces = [
        np.log1p(np.maximum(relative, 0)),
        np.log1p(
            np.maximum(relative, 0) /
            (0.001 + total[:, None] / max(typical, 1e-6))
        )
    ]

    for cols in bin_indices:
        if not cols:
            continue
        block = raw[:, cols]
        mass = block.sum(axis=1, keepdims=True)
        share = block / np.maximum(mass, 1)
        pieces.extend([
            share,
            np.cumsum(share, axis=1)[:, :-1],
            np.log1p(mass)
        ])

    return np.column_stack(pieces).astype("float32")


def utilization_score(features_matrix, fit, query, seed):
    days = train.followup_days.to_numpy()[fit]
    y = failure[fit]
    exposure = np.clip(days / 365.0, 0.08, 1.0)
    weights = np.where(y == 1, 6.0, exposure ** 2)

    model = HistGradientBoostingClassifier(
        max_leaf_nodes=10, learning_rate=0.055, max_iter=240,
        min_samples_leaf=35, max_bins=191, max_features=0.7,
        l2_regularization=7, early_stopping=False,
        random_state=seed
    )
    model.fit(features_matrix[fit], y, sample_weight=weights)
    return model.predict_proba(query)[:, 1]


def kernel_cost(raw, z, costs, raw_query, z_query):
    scales = np.maximum(np.nanquantile(raw, 0.75, axis=0), 1)

    def design(r, features_matrix):
        return np.nan_to_num(np.column_stack((
            features_matrix, np.isnan(features_matrix),
            features_matrix > 0, cost_basis(r, scales),
            channel_shapes(r)
        )), nan=0, posinf=0, neginf=0)

    scaler = StandardScaler()
    x = scaler.fit_transform(design(raw, z))
    y = np.log(costs)

    first = ElasticNet(
        alpha=0.065, l1_ratio=0.8, max_iter=5000
    ).fit(x, y)
    adaptive = np.maximum(
        np.sqrt(abs(first.coef_) / max(abs(first.coef_).max(), 1e-8)),
        0.075
    )
    second = ElasticNet(
        alpha=0.031, l1_ratio=0.75, max_iter=5000
    ).fit(x * adaptive, y)
    coefficients = second.coef_ * adaptive

    # Safe pooled kernel: a single group avoids assumptions about feature
    # dimensions after optional sensor filtering.
    loadings = coefficients.reshape(-1, 1)

    def coordinates(values):
        pooled = values @ loadings
        active = np.flatnonzero(abs(coefficients) > 0.014)
        detailed = (
            np.clip(values[:, active], -4, 4) *
            np.sqrt(abs(coefficients[active]))
        )
        return pooled, detailed

    def distances(a, b, pooled_train, detailed_train):
        return (
            cdist(a, pooled_train, "sqeuclidean"),
            cdist(b, detailed_train, "sqeuclidean"),
            a @ pooled_train.T
        )

    pooled_train, detailed_train = coordinates(x)
    train_dist = distances(
        pooled_train, detailed_train,
        pooled_train, detailed_train
    )

    identity = np.eye(len(y))
    centered = y - y.mean()

    def covariance(params, dist):
        d1, d2, linear = dist
        v = np.exp(params)
        return (
            v[0] * linear +
            v[1] * np.exp(-d1 / v[2]) +
            v[3] * np.exp(-d2 / v[4])
        )

    def factor(params):
        matrix = (
            covariance(params, train_dist) +
            np.exp(params[5]) * identity
        )
        return cho_factor(matrix, lower=True)

    def objective(params):
        chol = factor(params)
        return (
            0.5 * centered @ cho_solve(chol, centered) +
            np.log(np.diag(chol[0])).sum() +
            0.3 * np.sum(params[:5] ** 2)
        )

    start = np.log([1, .3, 4, .3, 6, .5])
    bounds = np.log([
        [.08, 4], [.005, 4], [.15, 40],
        [.005, 4], [.2, 80], [.23, 1.5]
    ])
    params = minimize(
        objective, start, bounds=bounds,
        method="L-BFGS-B", options={"maxiter": 45}
    ).x

    alpha = cho_solve(factor(params), centered)
    predictions = []

    for start_idx in range(0, len(raw_query), 1500):
        batch = slice(start_idx, start_idx + 1500)
        query_x = scaler.transform(
            design(raw_query[batch], z_query[batch])
        )
        pooled_q, detailed_q = coordinates(query_x)
        dist = distances(
            pooled_q, detailed_q,
            pooled_train, detailed_train
        )
        pred = y.mean() + covariance(params, dist) @ alpha
        predictions.append(np.clip(pred, y.min(), y.max()))

    return np.concatenate(predictions)


def interaction_cost(raw, z, costs, raw_query, z_query):
    scales = np.maximum(np.nanquantile(raw, 0.75, axis=0), 1)

    def design(readings, summaries):
        return np.nan_to_num(np.column_stack((
            summaries, np.isnan(summaries), summaries > 0,
            cost_basis(readings, scales),
            channel_shapes(readings)
        )), nan=0, posinf=0, neginf=0)

    scaler = StandardScaler()
    x = scaler.fit_transform(design(raw, z))
    y = np.log(costs)

    first = ElasticNet(
        alpha=0.061, l1_ratio=0.75, max_iter=5000
    ).fit(x, y)
    strength = np.abs(first.coef_)
    weights = np.maximum(
        np.sqrt(strength / max(strength.max(), 1e-8)),
        0.065
    )

    chosen = []
    for j in np.argsort(-strength):
        if strength[j] <= 0.004 or len(chosen) >= 26:
            break
        if not chosen or (
            np.max(np.abs(x[:, chosen].T @ x[:, j])) / len(x) < 0.86
        ):
            chosen.append(j)

    left, right = np.triu_indices(len(chosen), 1)
    pair_scaler = StandardScaler()

    def interactions(values):
        selected = np.clip(values[:, chosen], -3.5, 3.5)
        return np.column_stack((
            np.abs(selected),
            selected[:, left] * selected[:, right]
        ))

    combined = x * weights
    if chosen:
        pairs = pair_scaler.fit_transform(interactions(x))
        combined = np.column_stack((combined, 0.5 * pairs))

    second = ElasticNet(
        alpha=0.032, l1_ratio=0.75, max_iter=5000
    ).fit(combined, y)

    main_coef = second.coef_[:x.shape[1]] * weights
    pair_coef = second.coef_[x.shape[1]:] * 0.5
    predictions = []

    for start_idx in range(0, len(raw_query), 1500):
        batch = slice(start_idx, start_idx + 1500)
        q = scaler.transform(design(raw_query[batch], z_query[batch]))
        prediction = q @ main_coef
        if chosen:
            prediction += pair_scaler.transform(
                interactions(q)
            ) @ pair_coef
        predictions.append(prediction + second.intercept_)

    return np.clip(np.concatenate(predictions), y.min(), y.max())

## Inspection and parts-budget decisions

The policy is evaluated against the official cost:

\[
C(y,f,b)=
\begin{cases}
10+0.20\max(y-b,0)+0.02\max(b-y,0), & f=1,\\
y, & f=0,
\end{cases}
\]

where `y` is realised repair cost, `f` is the inspection flag, and `b` is the parts budget. For a non-failure, `y=0`, so inspection costs 10 plus the 0.02 return penalty on any unused budget. For a skipped inspection, the full realised repair cost is charged.

The optimal parts-budget quantile is adjusted for the probability of failure because a non-failing inspected vehicle has zero repair demand but still incurs the unused-budget penalty. The code uses the shortage and surplus rates in this calculation, rather than applying the usual failure-only newsvendor quantile.


In [ ]:
def estimate_prior(indices):
    old = X[indices, :len(stable)]
    new = X_test[:, :len(stable)]
    y = failure[indices].astype(bool)
    candidates = []

    for col, name in enumerate(stable):
        if shift[name] >= 0.06:
            continue
        positive, negative = old[y, col], old[~y, col]
        if np.isfinite(positive).sum() < 0.85 * y.sum():
            continue

        direction = (
            1 if np.nanmedian(positive) >= np.nanmedian(negative)
            else -1
        )
        cutoffs = np.nanquantile(
            direction * negative, [0.95, 0.97, 0.98, 0.99]
        )
        positive_tail = (
            direction * positive[:, None] > cutoffs
        ).mean(axis=0)
        negative_tail = (
            direction * negative[:, None] > cutoffs
        ).mean(axis=0)
        test_tail = (
            direction * new[:, col, None] > cutoffs
        ).mean(axis=0)

        contrast = positive_tail - negative_tail
        if contrast[0] >= 0.55:
            estimate = np.median(
                (test_tail - negative_tail) /
                np.maximum(contrast, 1e-6)
            )
            candidates.append((contrast[0], name, estimate))

    top = sorted(candidates, reverse=True)[:12]
    if len(top) < 3:
        return float(y.mean()), top
    return float(np.clip(
        np.median([estimate for _, _, estimate in top]),
        0.002, 0.15
    )), top


def prior_ratio(source, target):
    return (target / (1 - target)) / (source / (1 - source))


def vehicle_weights(actual, target, source):
    return np.where(
        actual > 0, target / source,
        (1 - target) / (1 - source)
    )


def individual_cost(actual, flag, budget):
    return np.where(
        flag,
        INSPECTION +
        SHORTAGE * np.maximum(actual - budget, 0) +
        SURPLUS * np.maximum(budget - actual, 0),
        actual
    )



# Small hand-checks against the official objective.
# No inspection: zero cost for a non-failure, full repair cost for a failure.
assert np.allclose(
    individual_cost(
        np.array([0.0, 100.0]), np.array([0, 0]), np.array([0.0, 0.0])
    ),
    np.array([0.0, 100.0])
)
# Inspection with no failure: inspection fee plus unused-budget penalty.
assert np.isclose(
    individual_cost(np.array([0.0]), np.array([1]), np.array([50.0]))[0],
    10.0 + 0.02 * 50.0
)
# Inspection with shortage and surplus.
assert np.isclose(
    individual_cost(np.array([100.0]), np.array([1]), np.array([80.0]))[0],
    10.0 + 0.20 * 20.0
)
assert np.isclose(
    individual_cost(np.array([100.0]), np.array([1]), np.array([120.0]))[0],
    10.0 + 0.02 * 20.0
)
print("Official objective unit checks passed.")


def risk_inputs(pred):
    risk_columns = [
        "raw_risk", "relative_risk", "kernel_risk"
    ]
    if "catboost_risk" in pred.columns:
        risk_columns.append("catboost_risk")

    scores = np.clip(
        pred[risk_columns].to_numpy(), 1e-6, 1 - 1e-6
    )
    logits = logit(scores)
    average = logits.mean(axis=1)

    return np.column_stack((
        logits,
        np.maximum(average + 4.5, 0),
        np.maximum(average + 1.5, 0)
    ))


def calibrated_risk(ref, query, actual, target_rate):
    a, b = risk_inputs(ref), risk_inputs(query)
    obs = actual > 0
    model = LogisticRegression(C=0.5, max_iter=300)
    model.fit(a, obs)
    recorded = model.predict_proba(b)[:, 1]
    recorded_target = expit(
        logit(np.clip(recorded, 1e-7, 1-1e-7)) +
        np.log(prior_ratio(obs.mean(), target_rate))
    )
    return recorded_target


def shape_demand(ref, query, actual, use_kernel=False):
    observed = actual > 0
    y = np.log(actual[observed])

    def severity(pred):
        sparse = pred[
            ["sparse_cost", "shape_cost", "adaptive_cost"]
        ].to_numpy() @ [0.15, 0.425, 0.425]
        if use_kernel and "kernel_cost" in pred.columns:
            return 0.5 * sparse + 0.5 * pred.kernel_cost.to_numpy()
        return sparse

    x = severity(ref[observed])
    slope = np.clip(
        ((x-x.mean()) @ (y-y.mean()) + 12) /
        ((x-x.mean()) @ (x-x.mean()) + 12),
        .65, 1.6
    )
    center = y.mean() + slope * (severity(query) - x.mean())
    errors = y - y.mean() - slope * (x-x.mean())
    return np.exp(np.clip(center, -4, 11)), np.sort(np.exp(errors)), actual.max()


def parts_decisions(p, demand, expected_raw, raw_to_target, direct_share):
    scale, residuals, ceiling = demand
    prefix = np.r_[0, np.cumsum(residuals)]
    n = len(residuals)

    cap = np.searchsorted(residuals, ceiling / scale, side="right")
    mean = (scale * prefix[cap] + (n-cap) * ceiling) / n

    q = 1 - SURPLUS / (
        (SHORTAGE + SURPLUS) * np.maximum(p, 1e-12)
    )
    index = np.clip(np.ceil(n*q).astype(int)-1, 0, n-1)
    budget = np.where(
        q > 0, np.minimum(scale * residuals[index], ceiling), 0.
    )

    below = np.searchsorted(residuals, budget/scale, side="right")
    returned = np.maximum(
        (budget*below - scale*prefix[below])/n, 0
    )
    shortage = np.maximum(mean - budget + returned, 0)

    inspect = (
        INSPECTION +
        SHORTAGE*p*shortage +
        SURPLUS*((1-p)*budget + p*returned)
    )

    expected = p * mean
    direct = np.clip(
        raw_to_target*expected_raw,
        expected/3, 3*expected
    )
    flag = (
        (1-direct_share)*expected +
        direct_share*direct > inspect
    )
    return flag.astype(int), np.where(flag, np.round(budget, 2), 0)


def incumbent_risk(ref, query, actual, prior):
    def boost_logits(frame):
        score = (
            0.45 * frame.raw_risk.to_numpy() +
            0.55 * frame.relative_risk.to_numpy()
        )
        return logit(
            np.clip(score, 1e-7, 1-1e-7)
        ).reshape(-1, 1)

    calibration = LogisticRegression(C=1).fit(
        boost_logits(ref), actual > 0
    )
    source_probability = calibration.predict_proba(
        boost_logits(query)
    )[:, 1]

    odds = prior_ratio((actual > 0).mean(), prior)
    target_probability = expit(
        logit(np.clip(source_probability, 1e-7, 1-1e-7)) +
        np.log(odds)
    )
    stacked = calibrated_risk(ref, query, actual, prior)
    risk = expit(
        logit(np.clip(0.7*stacked + 0.3*target_probability, 1e-7, 1-1e-7))
        + np.log(1.15)
    )
    return risk, source_probability, odds


def incumbent_decision(ref, query, actual, prior):
    risk, source_probability, _ = incumbent_risk(
        ref, query, actual, prior
    )
    demand = shape_demand(ref, query, actual)
    correction = risk / np.maximum(source_probability, 1e-7)
    return parts_decisions(
        risk, demand, query.direct_cost.to_numpy(),
        correction, 0.15
    )


def budget_inputs(budget, center):
    return np.column_stack((
        np.ones(len(budget)),
        np.clip(np.log(center / 350.0), -2, 2),
        np.clip(budget / center - 1.6, -1.5, 1.5)
    ))


def learn_budget_factors(budget, center, cost, prior):
    active = budget > 0
    x = budget_inputs(budget[active], center[active])
    target = cost[active] / budget[active]

    weights = vehicle_weights(
        cost, prior, (cost > 0).mean()
    )[active] * budget[active]
    weights /= weights.sum()

    quantile = SHORTAGE / (SHORTAGE + SURPLUS)
    penalty = np.array([0.03, 0.045, 0.045])

    def objective(coef):
        error = target - (1 + x @ coef)
        radius = np.hypot(error, 0.06)
        loss = (quantile - 0.5) * error + 0.5 * radius
        gradient = x.T @ (
            weights * (0.5 - quantile - 0.5*error/radius)
        )
        return (
            weights @ loss + 0.5 * penalty @ (coef**2),
            gradient + penalty * coef
        )

    fit = minimize(
        objective, np.zeros(3), method="L-BFGS-B", jac=True,
        bounds=[(-0.45, 0.55), (-0.35, 0.35), (-0.35, 0.35)]
    )
    if not fit.success or not np.isfinite(fit.x).all():
        raise ValueError(f"Budget calibration failed: {fit.message}")
    return fit.x


def apply_correction(budget, center, coef, ceiling):
    change = np.clip(
        budget_inputs(budget, center) @ coef, -0.4, 0.8
    )
    return np.minimum(
        ceiling, budget * (1 + change)
    ).round(2)


def maintenance_risk(ref, query, actual, prior):
    original_risk, source_probability, odds = incumbent_risk(
        ref, query, actual, prior
    )

    def inputs(frame):
        activity = logit(np.clip(
            frame.utilization_score.to_numpy(), 1e-8, 1-1e-8
        ))
        return np.column_stack((risk_inputs(frame), activity))

    model = LogisticRegression(C=0.5, max_iter=300).fit(
        inputs(ref), actual > 0
    )
    activity_risk = expit(
        model.decision_function(inputs(query)) +
        np.log(1.15 * odds)
    )
    risk = 0.6 * original_risk + 0.4 * activity_risk
    return risk, source_probability


def proposal(ref, query, actual, prior):
    risk, source_probability = maintenance_risk(
        ref, query, actual, prior
    )
    demand = shape_demand(
        ref, query, actual, use_kernel=True
    )
    correction = risk / np.maximum(source_probability, 1e-7)
    flags, budgets = parts_decisions(
        risk, demand, query.direct_cost.to_numpy(),
        correction, 0.15
    )
    return flags, budgets, demand[0]


def adjusted_decision(ref, query, actual, prior, estimate=proposal):
    training_budgets = np.empty(len(actual))
    training_centers = np.empty(len(actual))
    folds = StratifiedKFold(
        3, shuffle=True, random_state=77
    )

    for fit_idx, val_idx in folds.split(ref, actual > 0):
        _, training_budgets[val_idx], training_centers[val_idx] = estimate(
            ref.iloc[fit_idx], ref.iloc[val_idx],
            actual[fit_idx], prior
        )

    coefficients = learn_budget_factors(
        training_budgets, training_centers, actual, prior
    )
    flags, budgets, centers = estimate(
        ref, query, actual, prior
    )
    return flags, apply_correction(
        budgets, centers, coefficients, actual.max()
    )


def repair_score(frame):
    baseline_columns = [
        "sparse_cost", "shape_cost",
        "adaptive_cost", "kernel_cost"
    ]
    baseline = frame[baseline_columns].to_numpy()
    baseline_score = baseline @ [0.075, 0.2125, 0.2125, 0.5]
    if "interaction_cost" in frame.columns:
        return 0.7 * baseline_score + 0.3 * frame.interaction_cost.to_numpy()
    return baseline_score


def joint_risk_inputs(frame):
    scores = risk_inputs(frame)
    activity = logit(np.clip(
        frame.utilization_score.to_numpy(), 1e-7, 1-1e-7
    ))
    severity = frame[["adaptive_cost", "kernel_cost"]].mean(axis=1).to_numpy()
    level = np.clip(severity - np.log(200), -3, 3)
    direct = np.log(np.maximum(frame.direct_cost.to_numpy(), 1e-5))

    return np.column_stack((
        scores, activity, level,
        level * np.clip((scores[:, 0] + 4) / 4, -1, 1),
        np.clip(direct - scores[:, 0] - severity, -4, 4)
    ))


def local_proposal(ref, query, actual, prior):
    previous_risk, source = maintenance_risk(
        ref, query, actual, prior
    )
    observed = actual > 0
    odds = prior_ratio(observed.mean(), prior)

    model = LogisticRegression(C=0.35, max_iter=400).fit(
        joint_risk_inputs(ref), observed
    )
    calibrated = expit(
        model.decision_function(joint_risk_inputs(query)) +
        np.log(1.15 * odds)
    )
    risk = 0.35 * previous_risk + 0.65 * calibrated

    failures = ref[observed]
    log_cost = np.log(actual[observed])
    scores = repair_score(failures)
    centered = scores - scores.mean()

    slope = np.clip(
        (centered @ (log_cost-log_cost.mean()) + 12) /
        (centered @ centered + 12),
        0.65, 1.6
    )
    location = np.exp(np.clip(
        log_cost.mean() +
        slope * (repair_score(query)-scores.mean()),
        -4, 11
    ))
    errors = np.exp(
        log_cost - log_cost.mean() - slope*centered
    )
    order = np.argsort(errors)
    errors = errors[order]

    def coordinates(frame):
        return np.column_stack((
            repair_score(frame),
            risk_inputs(frame)[:, :3].mean(axis=1),
            frame.kernel_cost.to_numpy() -
            frame.sparse_cost.to_numpy()
        ))

    reference = coordinates(failures)[order]
    scale = np.maximum(reference.std(axis=0), 0.15) * [0.68, 1.1, 1.6]
    center = reference.mean(axis=0)
    reference = (reference-center) / scale
    positions = (coordinates(query)-center) / scale

    flags = np.empty(len(query), dtype=int)
    budgets = np.empty(len(query))

    for start_idx in range(0, len(query), 500):
        batch = slice(start_idx, start_idx+500)
        distance = cdist(
            positions[batch], reference, "sqeuclidean"
        )
        weights = np.exp(-0.5*np.minimum(distance, 80))
        weights += 25 / len(errors)
        weights /= weights.sum(axis=1, keepdims=True)
        weights = 0.85*weights + 0.15/len(errors)

        costs = np.minimum(
            location[batch, None]*errors, actual.max()
        )
        p = risk[batch]
        quantile = 1 - SURPLUS / (
            (SHORTAGE+SURPLUS)*np.maximum(p, 1e-12)
        )
        ix = np.minimum(
            (np.cumsum(weights, axis=1) < quantile[:, None]).sum(axis=1),
            len(errors)-1
        )
        parts = np.where(
            quantile > 0, costs[np.arange(len(p)), ix], 0
        )

        expected = p*(costs*weights).sum(axis=1)
        direct = (
            query.direct_cost.to_numpy()[batch] *
            p/np.maximum(source[batch], 1e-7)
        )
        skip = 0.85*expected + 0.15*np.clip(
            direct, expected/3, 3*expected
        )

        shortage = (
            np.maximum(costs-parts[:, None], 0)*weights
        ).sum(axis=1)
        surplus = (
            np.maximum(parts[:, None]-costs, 0)*weights
        ).sum(axis=1)

        inspect = (
            INSPECTION +
            SHORTAGE*p*shortage +
            SURPLUS*((1-p)*parts+p*surplus)
        )

        flags[batch] = skip > inspect
        budgets[batch] = np.where(
            flags[batch], parts.round(2), 0
        )

    return flags, budgets, location


def maintenance_decision(ref, query, actual, prior):
    return adjusted_decision(
        ref, query, actual, prior, estimate=local_proposal
    )

## Cross-validation and model comparison

This section creates OOF predictions and test predictions using the original
five-fold structure. The CatBoost risk and repair-cost models are optional
challengers. The CatBoost repair-cost prediction is retained as a separate column
so it can be evaluated before it is allowed to influence the maintenance policy.

**Validation caveat:** the original second-stage policy validation uses OOF rows
whose base models may have been trained on other outer-validation rows. This can
produce optimistic policy-validation estimates. A fully nested evaluation is
recommended before treating small score differences as reliable.

In [ ]:
def feature_predictions(fit_idx, query_source, query_idx, seed):
    """Fit all base models on fit_idx and predict a train or test query slice."""
    pos = fit_idx[failure[fit_idx] == 1]
    if query_source == "train":
        z_query = Z[query_idx]
        raw_query = raw_cost[query_idx]
        x_query = X[query_idx]
        activity_query = activity_train[query_idx]
    elif query_source == "test":
        z_query, raw_query = Z_test, test_cost
        x_query, activity_query = X_test, activity_test
    else:
        raise ValueError("query_source must be 'train' or 'test'")

    blocks = [
        fit_predict(fit_idx, x_query, seed),
        relative_predictions(fit_idx, z_query, seed),
        kernel_risk(fit_idx, z_query, SEED + 8 + seed).reshape(-1, 1),
        fit_cost_models(
            raw_cost[pos], Z[pos], repair[pos], raw_query, z_query
        ),
        utilization_score(
            activity_train, fit_idx, activity_query, seed
        ).reshape(-1, 1),
        kernel_cost(
            raw_cost[pos], Z[pos], repair[pos], raw_query, z_query
        ).reshape(-1, 1),
        interaction_cost(
            raw_cost[pos], Z[pos], repair[pos], raw_query, z_query
        ).reshape(-1, 1)
    ]
    if USE_CATBOOST:
        blocks.extend([
            catboost_risk(fit_idx, z_query, seed).reshape(-1, 1),
            catboost_repair_cost(
                raw_cost[pos], repair[pos], raw_query, seed
            ).reshape(-1, 1)
        ])

    # Conditional repair-cost quantiles, trained only on observed failures.
    # GradientBoostingRegressor with quantile loss is explicitly permitted.
    quantile_levels = [0.05, 0.15, 0.35, 0.625, 0.825, 0.925, 0.975]
    quantile_predictions = []

    def quantile_design(raw):
        # GradientBoostingRegressor does not accept NaNs; preserve missingness
        # explicitly and replace missing sensor values with zero.
        values = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
        missing = np.isnan(raw).astype(float)
        return np.column_stack((values, missing))

    positive_log_cost = np.log(np.maximum(repair[pos], 1.0))
    quantile_x = quantile_design(raw_cost[pos])
    quantile_query = quantile_design(raw_query)

    for quantile_index, alpha in enumerate(quantile_levels):
        quantile_model = HistGradientBoostingRegressor(
            loss="quantile", quantile=alpha,
            max_iter=140, learning_rate=0.04,
            max_leaf_nodes=7, min_samples_leaf=10,
            l2_regularization=5.0, max_bins=127,
            early_stopping=False,
            random_state=seed + 700 + quantile_index
        )
        quantile_model.fit(quantile_x, positive_log_cost)
        predicted_log_quantile = quantile_model.predict(quantile_query)
        quantile_predictions.append(
            np.exp(np.clip(predicted_log_quantile, 0, np.log(repair.max())))
        )
    blocks.append(np.column_stack(quantile_predictions))

    result = np.column_stack(blocks)
    if result.shape[1] != len(names):
        raise ValueError(f"Feature block shape {result.shape}; expected {len(names)} columns")
    if not np.isfinite(result).all():
        raise ValueError("Non-finite base-model predictions encountered")
    return result

raw_cost = cost_readings(train, source_channels)
test_cost = cost_readings(test, target_channels)

activity_train = utilization_features(raw_cost)
activity_test = utilization_features(test_cost)

names = [
    "raw_risk", "direct_cost",
    "relative_risk", "sparse_cost",
    "kernel_risk",
    "shape_cost", "adaptive_cost",
    "utilization_score", "kernel_cost",
    "interaction_cost"
]
if USE_CATBOOST:
    names += ["catboost_risk", "catboost_cost"]

QUANTILE_LEVELS = [0.05, 0.15, 0.35, 0.625, 0.825, 0.925, 0.975]
QUANTILE_COLUMNS = [f"repair_q{int(q * 1000):03d}" for q in QUANTILE_LEVELS]
names += QUANTILE_COLUMNS

indices = np.arange(len(train))
folds = list(
    StratifiedKFold(
        5, shuffle=True, random_state=SEED+1
    ).split(indices, failure)
)

oof = pd.DataFrame(
    np.nan, index=indices, columns=names
)
test_folds = []

for fold, (fit_idx, val_idx) in enumerate(folds, 1):
    seed = SEED + 1 + fold
    oof.iloc[val_idx] = feature_predictions(
        fit_idx, "train", val_idx, seed
    )
    test_folds.append(
        feature_predictions(fit_idx, "test", None, seed)
    )
    print(f"Finished final OOF fold {fold}/5", flush=True)

test_scores = pd.DataFrame(
    np.mean(test_folds, axis=0), columns=names
)

assert np.isfinite(oof.to_numpy()).all()
assert np.isfinite(test_scores.to_numpy()).all()

prior, prior_details = estimate_prior(indices)
print(f"Estimated new-depot failure rate: {prior:.4%}")
print(f"CatBoost enabled: {USE_CATBOOST}")

## New experiment: direct expected-cost policy

This experiment makes the inspection decision and parts-budget decision jointly.

- Failure probabilities are calibrated from out-of-fold risk predictions.
- Conditional repair-cost quantiles are predicted using
  `sklearn.ensemble.HistGradientBoostingRegressor` with quantile loss.
- A weighted quantile grid approximates the conditional repair-demand distribution.
- For each vehicle, the notebook evaluates the expected cost of skipping inspection
  and the expected cost of inspection at candidate budgets.
- It chooses the lower-cost action using the official shortage, surplus, and
  inspection penalties.

The quantile models are fitted only on training-fold vehicles with observed failures.
Their predictions for validation/test vehicles are generated without fitting on those
queried rows. The policy is included in the nested holdout comparison and is selected
only if its average validation cost is lower than the alternatives.


In [ ]:
def direct_expected_cost_decision(ref, query, actual, prior):
    """
    Jointly choose inspection and parts budget by minimizing the
    competition's expected cost for each vehicle.

    Failure risk is calibrated from cross-fitted risk predictions.
    Conditional repair demand is approximated by a weighted set of
    cross-fitted quantile predictions from observed failures only.
    """
    p = np.clip(
        calibrated_risk(ref, query, actual, prior),
        1e-7, 1 - 1e-7
    )

    quantile_values = query[QUANTILE_COLUMNS].to_numpy(dtype=float)
    quantile_values = np.maximum(quantile_values, 0.0)
    quantile_values.sort(axis=1)

    # Representative probability masses for the quantile grid.
    demand_weights = np.array(
        [0.10, 0.15, 0.25, 0.25, 0.15, 0.07, 0.03],
        dtype=float
    )
    demand_weights /= demand_weights.sum()

    # Candidate budgets include zero and every predicted conditional
    # quantile. Evaluating the candidates directly avoids a heuristic
    # probability threshold or a separate severity blending rule.
    candidate_budgets = np.column_stack((
        np.zeros(len(query)),
        quantile_values
    ))

    conditional_mean = quantile_values @ demand_weights
    skip_cost = p * conditional_mean

    best_inspect_cost = np.full(len(query), np.inf)
    best_budget = np.zeros(len(query), dtype=float)

    for candidate_index in range(candidate_budgets.shape[1]):
        budget = candidate_budgets[:, candidate_index]
        shortage = (
            np.maximum(quantile_values - budget[:, None], 0.0)
            @ demand_weights
        )
        conditional_surplus = (
            np.maximum(budget[:, None] - quantile_values, 0.0)
            @ demand_weights
        )

        inspect_cost = (
            INSPECTION
            + SHORTAGE * p * shortage
            + SURPLUS * (
                (1.0 - p) * budget
                + p * conditional_surplus
            )
        )

        improve = inspect_cost < best_inspect_cost
        best_inspect_cost[improve] = inspect_cost[improve]
        best_budget[improve] = budget[improve]

    inspect_flag = best_inspect_cost < skip_cost
    final_budget = np.where(inspect_flag, best_budget, 0.0)

    return (
        inspect_flag.astype(int),
        np.round(final_budget, 2)
    )


print("Direct expected-cost inspection and quantile-budget policy ready.")


## Optional challenger ablation

Before changing the maintenance policy, compare CatBoost's risk predictions
against the existing risk ensemble. This is not a substitute for the economic
validation below, but it helps identify a model that contributes little new
information.

In [ ]:
if USE_CATBOOST:
    from sklearn.metrics import roc_auc_score, log_loss

    y_true = failure
    existing = oof[
        ["raw_risk", "relative_risk", "kernel_risk"]
    ].mean(axis=1).clip(1e-6, 1-1e-6)
    cat = oof["catboost_risk"].clip(1e-6, 1-1e-6)

    print("Existing risk ensemble:")
    print("  ROC-AUC:", round(roc_auc_score(y_true, existing), 5))
    print("  Log loss:", round(log_loss(y_true, existing), 5))

    print("CatBoost risk:")
    print("  ROC-AUC:", round(roc_auc_score(y_true, cat), 5))
    print("  Log loss:", round(log_loss(y_true, cat), 5))

    observed = repair > 0
    y_cost = np.log(repair[observed])
    base_cost = oof.loc[observed, [
        "sparse_cost", "shape_cost", "adaptive_cost"
    ]].mean(axis=1)
    cat_cost = oof.loc[observed, "catboost_cost"]

    print("Repair log-cost MAE:")
    print("  Existing severity average:",
          round(np.mean(np.abs(base_cost-y_cost)), 5))
    print("  CatBoost severity:",
          round(np.mean(np.abs(cat_cost-y_cost)), 5))

## Maintenance-policy validation

The three original policies are retained. The optional CatBoost features are
available to calibration through `risk_inputs`, but the CatBoost repair-cost
prediction is not automatically mixed into the existing severity score. This
avoids changing several components at once.

In [ ]:
# Nested/cross-fitted policy validation on one stratified holdout.
# The meta-model's training rows are built from inner-fold predictions; the
# outer validation rows are predicted by base models fitted only on outer_train.
from sklearn.model_selection import train_test_split

outer_train, outer_val = train_test_split(
    indices, test_size=0.20, random_state=SEED + 202,
    stratify=failure
)
inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED + 303)
ref_oof = pd.DataFrame(np.nan, index=outer_train, columns=names)

for inner_fold, (inner_fit_local, inner_val_local) in enumerate(
    inner.split(outer_train, failure[outer_train]), 1
):
    inner_fit = outer_train[inner_fit_local]
    inner_val = outer_train[inner_val_local]
    ref_oof.loc[inner_val] = feature_predictions(
        inner_fit, "train", inner_val, SEED + 400 + inner_fold
    )
    print(f"Nested policy-training fold {inner_fold}/4", flush=True)

outer_query = pd.DataFrame(
    feature_predictions(
        outer_train, "train", outer_val, SEED + 505
    ), index=outer_val, columns=names
)
assert np.isfinite(ref_oof.to_numpy()).all()
assert np.isfinite(outer_query.to_numpy()).all()

fold_prior, prior_details = estimate_prior(outer_train)
policies = {
    "personal best": incumbent_decision,
    "previous revision": adjusted_decision,
    "local repair costs": maintenance_decision,
    "direct expected cost + quantiles": direct_expected_cost_decision
}

rows = []
for rate in fold_prior * np.array([0.65, 1.0, 1.35]):
    weights = vehicle_weights(
        repair[outer_val], rate, failure[outer_train].mean()
    )
    policy_costs = {}
    for label, make_decision in policies.items():
        flag, budget = make_decision(
            ref_oof, outer_query, repair[outer_train], rate
        )
        policy_costs[label] = float(np.average(
            individual_cost(repair[outer_val], flag, budget),
            weights=weights
        ))
    rows.append({"assumed failure rate": rate, **policy_costs})

comparison = pd.DataFrame(rows)
print("Nested holdout policy cost (lower is better):")
print(comparison.round(4).to_string(index=False))
print("\nUse this as a more honest policy comparison than the former OOF-on-OOF evaluation.")
print("It is still one holdout; repeat with another seed before trusting small differences.")

# Select the policy with the lowest average nested-holdout cost across the
# three prior-rate scenarios. Repeated holdouts are recommended if costs are close.
policy_names = list(policies)
mean_policy_cost = comparison[policy_names].mean(axis=0)
selected_policy_name = mean_policy_cost.idxmin()
selected_policy = policies[selected_policy_name]
print("Selected policy from nested holdout:", selected_policy_name)
print("Mean holdout costs:", mean_policy_cost.round(4).to_dict())


## Kaggle submission

The selected policy is the original `local repair costs` policy, now using the
revised feature pipeline and optional CatBoost risk challenger. The submission
schema and row order are checked before the SHA-256 fingerprint is printed.

In [ ]:
flag, budget = selected_policy(
    oof, test_scores, repair, prior
)

print(f"Submission policy: {selected_policy_name}")

predictions = pd.DataFrame({
    "id": test.id,
    "flag": flag,
    "budget": budget
}).set_index("id")

submission = predictions.loc[sample.id].reset_index()

output = "submission.csv"
submission.to_csv(output, index=False)

written = pd.read_csv(output, dtype={"id": str})

assert written.columns.tolist() == SCHEMA
assert len(written) == 20_000
assert written.id.is_unique
assert written.id.equals(sample.id)
assert written.flag.isin([0, 1]).all()
assert np.isfinite(written.budget).all()
assert written.budget.ge(0).all()
assert written.loc[written.flag.eq(0), "budget"].eq(0).all()

print(f"Saved {len(written):,} rows to {output}")
print(
    f"Inspections: {written.flag.sum():,}. "
    f"Positive budgets: {written.budget.gt(0).sum():,}."
)

for label, previous_policy in list(policies.items())[:2]:
    old_flag, old_budget = previous_policy(
        oof, test_scores, repair, prior
    )
    changes = (
        (flag != old_flag) |
        (budget != old_budget)
    )
    print(f"Rows differing from {label}: {changes.sum():,}")



## Competition-rule compliance checklist
- Models used are from the permitted list: HistGradientBoosting, ElasticNet, logistic regression, Nystroem/kernel methods, and optional CatBoost.
- Third-party packages used are from the permitted list: NumPy, pandas, SciPy, scikit-learn, and optional CatBoost. `pathlib`, `hashlib`, and `warnings` are Python standard-library modules; ask the TA if the rule's wording is intended to restrict standard-library modules too.
- No external data, source-identification, pretrained models, deep-learning frameworks, AutoML, generated labels, or manual test labels are used.
- Test-set feature distributions are used only for unsupervised sensor-shift/alignment diagnostics and the prior heuristic. These use competition-provided unlabeled features, not outside data or test labels. Confirm with the TA if they require a stricter train-only protocol.
- Do not privately share competition code or predictions outside your registered team.

**The supplied competition description confirms the cost coefficients used above.** The meaning of `followup_days` still needs verification before treating short-follow-up non-failures as full-horizon negatives.

- The added conditional repair quantiles use `sklearn.ensemble.HistGradientBoostingRegressor` with quantile loss, which is within the permitted scikit-learn gradient-boosting family.
- The direct expected-cost policy uses NumPy calculations and the official competition cost constants; it does not use an external package.


## Final run checklist

1. Place `train.csv`, `test.csv`, and `sample_submission.csv` in one dataset directory and set `DATA_DIR` in the setup cell to that directory.
2. Run the notebook from the first cell to the last without skipping cells.
3. Confirm that the follow-up audit is understood. If short-follow-up non-failures are right-censored, the current binary classification treatment needs revision before relying on its risk estimates.
4. Check the nested-holdout maintenance-cost table. Compare policies on the same folds/seeds; small differences from one holdout are not reliable.
5. Confirm the printed estimated new-depot failure rate is plausible. It is a sensor-based prior-shift heuristic, not a measured test failure rate.
6. Confirm CatBoost is enabled or disabled as expected. It is optional and permitted by the supplied rules.
7. Confirm the final output is `submission.csv`, has exactly 20,000 rows and columns `id, flag, budget`, preserves sample-submission order, uses only flags 0/1, and has non-negative budgets (zero when `flag=0`).

The notebook validates the file format, but only a successful full run with the actual competition files can establish that the submission file was generated correctly. No validation score or leaderboard improvement is guaranteed.
